# BTS Digital Twin - Train Or Resume

Notebook này chỉ phục vụ 1 workflow:

- chạy từ đầu ở `30000` iteration
- lưu `gs_model/` đầy đủ, gồm cả `.pth`
- lần sau dán link Google Drive để resume lên `60000` hoặc cao hơn


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
!pip install -q gdown plyfile tqdm


## Bước 1 - Clone gaussian-splatting


In [ ]:
%cd /kaggle/working
!rm -rf gaussian-splatting
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /kaggle/working/gaussian-splatting
!git checkout 54c035f7834b564019656c3e3fcc3646292f727d
!git submodule update --init --recursive
!pip install -q ./submodules/diff-gaussian-rasterization
!pip install -q ./submodules/simple-knn
import os
os.environ['GS_REPO'] = '/kaggle/working/gaussian-splatting'


## Bước 2 - Clone repo pipeline


In [ ]:
REPO_URL = 'https://github.com/ThongLuc2k3/BTS-Digital-Twin.git'
GIT_BRANCH = 'main'
GITHUB_TOKEN = ''

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
        print('Đã lấy GITHUB_TOKEN từ Kaggle Secrets')
except Exception:
    pass

clone_url = REPO_URL
if GITHUB_TOKEN and 'github.com' in REPO_URL:
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')

!rm -rf /kaggle/working/project
!git clone --depth 1 -b "{GIT_BRANCH}" "{clone_url}" /kaggle/working/project
%cd /kaggle/working/project


## Bước 3 - Dataset

Nếu dataset đã mount sẵn trên Kaggle, để `USE_DRIVE_DATASET = False`.
Nếu chưa có, bật `USE_DRIVE_DATASET = True` và dán link zip dataset.


In [ ]:
USE_DRIVE_DATASET = False
DATASET_DRIVE_URL = ''
DATASET_ROOT = '/kaggle/working/project/Dataset/VAI_NVS_DATA_ROUND2'

if USE_DRIVE_DATASET:
    assert DATASET_DRIVE_URL, 'Chưa điền DATASET_DRIVE_URL'
    !mkdir -p /kaggle/working/_dataset_raw
    !gdown --fuzzy "{DATASET_DRIVE_URL}" -O /kaggle/working/dataset.zip
    !unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/project/Dataset

import os
assert os.path.isdir(DATASET_ROOT), f'Không thấy DATASET_ROOT: {DATASET_ROOT}'
os.environ['DATASET_ROOT'] = DATASET_ROOT
print('DATASET_ROOT =', DATASET_ROOT)


## Bước 4 - Cấu hình train hoặc resume


In [ ]:
MODE = 'train'  # train | resume
SCENE = 'HCM0421'
ITERATIONS = '30000'
CHECKPOINT_DRIVE_LINK = ''
ANTIALIASING = '1'
EXPOSURE_COMP = '1'
SAVE_FINAL_CHECKPOINT = '1'

assert MODE in {'train', 'resume'}
if MODE == 'resume':
    assert CHECKPOINT_DRIVE_LINK, 'MODE=resume nhưng chưa điền CHECKPOINT_DRIVE_LINK'

print('MODE =', MODE)
print('SCENE =', SCENE)
print('ITERATIONS =', ITERATIONS)


## Bước 5 - Tải checkpoint cũ nếu resume


In [ ]:
import shutil
from pathlib import Path
import os

model_dir = Path(f'/kaggle/working/project/pipeline/work/{SCENE}/gs_model')
if MODE == 'resume':
    raw_dl_dir = Path(f'/kaggle/working/_ckpt_raw/{SCENE}')
    shutil.rmtree(raw_dl_dir, ignore_errors=True)
    raw_dl_dir.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(model_dir, ignore_errors=True)
    print(f'===== {SCENE}: tải gs_model từ Drive =====')
    !gdown --fuzzy --folder "{CHECKPOINT_DRIVE_LINK}" -O "{raw_dl_dir}"
    candidates = [p.parent for p in raw_dl_dir.rglob('cfg_args')]
    assert candidates, 'Không tìm thấy cfg_args trong thư mục tải về'
    src_root = candidates[0]
    shutil.copytree(src_root, model_dir)
    ckpts = sorted(model_dir.glob('chkpnt*.pth'))
    assert ckpts, 'Không có file chkpnt*.pth để resume'
    os.environ['START_CHECKPOINT'] = str(ckpts[-1])
    print('START_CHECKPOINT =', os.environ['START_CHECKPOINT'])
else:
    os.environ.pop('START_CHECKPOINT', None)


## Bước 6 - Chạy train hoặc resume


In [ ]:
import os
os.environ['ITERATIONS'] = ITERATIONS
os.environ['ANTIALIASING'] = ANTIALIASING
os.environ['EXPOSURE_COMP'] = EXPOSURE_COMP
os.environ['SAVE_FINAL_CHECKPOINT'] = SAVE_FINAL_CHECKPOINT

!bash /kaggle/working/project/pipeline/scripts/03_train_3dgs.sh {SCENE}


## Bước 7 - File phải giữ lại

Sau khi xong, lấy nguyên thư mục này về rồi upload lại lên Google Drive nếu muốn resume tiếp lần sau:

`/kaggle/working/project/pipeline/work/<SCENE>/gs_model`


In [ ]:
print(f'/kaggle/working/project/pipeline/work/{SCENE}/gs_model')
